# S04 — Perception visuelle : choisir le bon graphique

Un même tableau de chiffres peut conduire à des conclusions très différentes selon la façon dont on le dessine.
Aujourd'hui, nous allons :

1. mesurer ce que l'œil lit bien (et mal) ;
2. diagnostiquer trois graphiques ratés construits sur nos données NYC Taxi ;
3. les refaire correctement ;
4. apprendre à choisir le graphique à partir de la question posée.

Prérequis : les tables `gold_trajets_horaires` et `gold_zones` existent dans PostgreSQL (séance S02).

In [1]:
import sys; sys.path.append("/home/jovyan/work")
from commun.donnees import lire

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

horaires = lire("gold_trajets_horaires")
zones = lire("gold_zones")
horaires["heure_pickup"] = pd.to_datetime(horaires["heure_pickup"])
horaires["heure"] = horaires["heure_pickup"].dt.hour
arr = (horaires[~horaires["arrondissement"].isin(["Unknown", "N/A"])]
       .groupby("arrondissement", as_index=False)
       .agg(courses=("courses", "sum"), revenu=("revenu", "sum")))
arr["prix_moyen"] = arr["revenu"] / arr["courses"]
arr

ProgrammingError: (psycopg2.errors.UndefinedTable) relation "gold_trajets_horaires" does not exist
LINE 1: SELECT * FROM gold_trajets_horaires
                      ^

[SQL: SELECT * FROM gold_trajets_horaires]
(Background on this error at: https://sqlalche.me/e/21/f405)

## 1. Ce que l'œil lit bien

Cleveland et McGill (1984) ont mesuré la précision avec laquelle on compare des quantités selon leur encodage visuel.
Du plus précis au moins précis :

1. **position sur une échelle commune** (barres alignées, points) ;
2. position sur des échelles non alignées ;
3. **longueur** ;
4. angle, pente ;
5. **aire** ;
6. volume, saturation et teinte de couleur.

Testez-le : sans lire les nombres, classez les cinq valeurs ci-dessous sur le camembert, puis sur les barres.

In [ ]:
valeurs = pd.Series([23, 21, 19, 20, 17], index=list("ABCDE"))
fig, (g, d) = plt.subplots(1, 2, figsize=(11, 4))
g.pie(valeurs, labels=valeurs.index, startangle=90); g.set_title("Camembert")
d.bar(valeurs.index, valeurs); d.set_title("Barres")
plt.show()

Sur le camembert, l'ordre est presque impossible à lire. Sur les barres, il saute aux yeux.
C'est la différence entre un angle et une position sur une échelle commune.

## 2. Atelier : trois graphiques ratés

Pour chacun des trois graphiques :

- écrivez **la question** à laquelle il devrait répondre ;
- listez **tout ce qui ne va pas** (type de graphique, axes, couleurs, titre, unités…) ;
- **refaites-le** dans la cellule prévue, puis justifiez chaque choix en une phrase.

In [ ]:
# Graphique A
plt.figure(figsize=(6, 6))
plt.pie(arr["prix_moyen"], labels=arr["arrondissement"], autopct="%1.0f%%",
        explode=[0.05] * len(arr), shadow=True, startangle=90)
plt.title("Prix")
plt.show()

**Graphique A**

- Question visée : …
- Ce qui ne va pas : …

In [ ]:
# Votre version du graphique A

In [ ]:
# Graphique B
couleurs = plt.cm.jet(np.linspace(0, 1, len(arr)))
plt.figure(figsize=(8, 4))
plt.bar(arr["arrondissement"], arr["courses"], color=couleurs)
plt.ylim(arr["courses"].min() * 0.95, arr["courses"].max() * 1.01)
plt.title("Courses")
plt.show()

**Graphique B**

- Question visée : …
- Ce qui ne va pas : …

In [ ]:
# Votre version du graphique B

In [ ]:
# Graphique C
par_heure = horaires.groupby("heure").agg(courses=("courses", "sum"), temperature=("temperature_c", "mean"))
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(par_heure.index, par_heure["courses"], color="red")
ax2 = ax1.twinx()
ax2.plot(par_heure.index, par_heure["temperature"], color="green")
ax2.set_ylim(par_heure["temperature"].min() - 0.2, par_heure["temperature"].max() + 0.2)
plt.title("Courses et température")
plt.show()

**Graphique C**

- Question visée : …
- Ce qui ne va pas : …

In [ ]:
# Votre version du graphique C

## 3. Choisir le graphique à partir de la question

| La question porte sur… | Graphique adapté | À éviter |
|---|---|---|
| une **comparaison** entre catégories | barres horizontales triées, points | camembert à plus de 3 parts, 3D |
| une **évolution** dans le temps | courbe, aires | barres très nombreuses, axe non chronologique |
| une **répartition** d'une variable | histogramme, boîte à moustaches | moyenne seule |
| une **relation** entre deux variables | nuage de points (avec agrégation si gros volume) | double axe |
| une **part d'un tout** (peu de parts) | barre empilée à 100 %, camembert à 2–3 parts | camembert à 8 parts |
| une **géographie** | carte choroplèthe (valeurs normalisées) | carte de valeurs brutes non rapportées |

**Exercice :** pour chacune des questions suivantes, choisissez un graphique, construisez-le et donnez-lui un titre qui énonce la conclusion
(pas seulement le sujet).

1. À quelle heure la demande est-elle la plus forte ?
2. Quelles sont les 10 zones de départ où le prix moyen est le plus élevé ?
3. La distance moyenne des courses varie-t-elle selon l'heure ?

In [ ]:
# Question 1

In [ ]:
# Question 2

In [ ]:
# Question 3

## 4. La couleur

- **Séquentielle** (du clair au foncé) pour une quantité qui va de faible à fort : `viridis`, `Blues`.
- **Divergente** (deux teintes autour d'un centre) pour un écart à une référence : `RdBu`, `PuOr`.
- **Catégorielle** (teintes distinctes) pour des groupes sans ordre : `tab10`, `Set2`.
- À proscrire : `jet` et les arcs-en-ciel, qui créent des frontières qui n'existent pas dans les données
  et sont illisibles pour une partie des daltoniens.

In [ ]:
x = np.linspace(0, 1, 256).reshape(1, -1)
fig, axes = plt.subplots(3, 1, figsize=(8, 2.4))
for ax, nom in zip(axes, ["jet", "viridis", "RdBu"]):
    ax.imshow(x, aspect="auto", cmap=nom); ax.set_axis_off(); ax.set_title(nom, fontsize=9, loc="left")
plt.tight_layout(); plt.show()

## À retenir

- On part toujours de **la question**, jamais de l'outil.
- On privilégie la **position sur une échelle commune** ; on se méfie des angles, des aires et de la 3D.
- Un axe de barres commence à **zéro**.
- Le titre énonce **la conclusion** ; les axes ont des **unités**.
- La couleur a un **sens** (séquentielle, divergente, catégorielle) ou n'est pas utilisée.